In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (PeptiTox)

This notebook curates the **PeptiTox** dataset from a pre-merged CSV file. The pipeline performs a final standardization pass, checks duplicated sequences for label consistency, builds dataset metadata, and exports a clean toxic peptide dataset for downstream modeling.

- **Toxic effect / endpoint:** toxic
- **Source:** PeptiTox
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the curated/merged input table** (`merged_data.csv`) containing at minimum:
  - `sequence`
  - `label`
- **Checks duplicated sequences**:
  - identical sequences with consistent labels are collapsed,
  - sequences with conflicting labels are flagged and exported as errors.
- **Builds metadata** using the project-wide Excel description sheet and appends dataset-level QC statistics.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`,
  - `metadata.json`.

In [2]:
name_source = "PeptiTox"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df = pd.read_csv(f"{PATH_INPUT}/{name_source}/merged_data.csv")
df.shape

(3864, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(3864, 2)

In [6]:
df_errors.shape

(0, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 3, 28, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://github.com/WYLqlit/PeptiTox/blob/main/Data/merged_data.csv',
 'publication': 'https://pubs.acs.org/doi/full/10.1021/acs.jcim.5c01073',
 'number_of_raw_sequences': 3864,
 'number_of_sequences_retained': 3864,
 'number_of_positive_sequences': 1932,
 'number_of_negative_sequences': 1932,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)